# **Smartphone Behaviour and Student Stress: A CRISP-DM Analysis of StudentLife**

## DBMS71008 Data Analytics | CA2 CRISP-DM Analytics Project | Daniel Courtney

## **Phase 1: Business Understanding**

### **1.1 Context and Problem**

University students face recurring periods of high academic pressure, and stress tends to rise as workload increases across the academic term. Heavy smartphone use and poor sleep are commonly suggested as contributing factors. However, student support services typically learn about difficulties only when students self report them, after problems have escalated, with much of this stress remaining hidden from staff.

Smartphones offer a way to observe daily behaviour. This project investigates whether behavioural signals captured by smartphones are associated with students self reported stress, using the *StudentLife dataset*. *(Wang et al, 2014)*


### **1.2 Stakeholders and Users**

| Stakeholder | Role | Interest / How they are affected |
| --- | --- | --- |
| StudentLife Participants | Supply data | Behaviour and stress data analysed for improved supports |
| Student Support Services | Primary User | Want insights into when students may need support, to guide outreach and resources |
| StudentLife Research Team | Data Owner | Conduct ethical and insightful data on student life |
| Researchers | Secondary Users | Interested in whether sensing can support wellbeing research |
| Data Protection Officer | Governance | Decide whether monitoring is acceptable, and under what consent, privacy, and data protection rules. |

### **1.3 Business or Practical Objective**

**Practical Objective:**

Help student support services understand which daily behaviours are associated with elevated stress, so they can decide when and how to offer proactive support.

### **1.4 Analytics Objective**

**Question**

To what extent can passively sensed smartphone usage and sleep behaviour distinguish days on which university students report elevated stress from days on which they do not?


**Analytics Objective**

Build a binary classifier (model to sort row into one of two groups) that distinguishes high stress from other student days using daily phone use, sleep, and activity features (inputs model will use to make a decision), and evaluate whether it outperforms a majority-class baseline (predict whichever class is most common) in recall and F1 (single number that balances precision and recall) for high-stress days on students not seen during training.

Recall is essential as it means support services missing a struggling student (false negative) is wose than an unnecessary check in (false positive) *(Adler et al, 2012)*. 

### **1.5 Success Criteria**

**Technical Criteria**

| Criterion | How it will be measured |
| --- | --- |
| Beats the baseline | Recall and F1 for high-stress days exceed a majority-class baseline |
| Generalises to new students | Evaluated on students held out from training |

<br>

**Practical Criteria**

| Criterion | How it will be measured |
| --- | --- |
| Interpretable | Main patterns can be explained in plain language to non-technical support staff |
| Useful for the intended decision | Findings indicate which behaviours or periods relate to elevated stress, informing outreach timing |
| Appropriate use | Results describe patterns, not labels on individual students |
| Honest limitations | Sample size, single institution and 2013 data clearly stated |

### **1.6 Constraints and Assumptions**

**Constraints**

| Type | Constraint |
| --- | --- |
| Time | Individual project, with final submission due 4 December 2026 |
| Data | Limited to the existing StudentLife data (48 students, one class, Spring 2013), no new data can be collected |
| Scope | Stress as the only outcome. Phone use, sleep and activity as the only feature areas. Location data excluded |

<br>

**Assumptions**

| Assumption | Why it matters |
| --- | --- |
| Self-reported stress reflects how stressed students actually felt | It is the target label. If it is inaccurate, the model learns noise |
| Sensor inferences (activity, sleep) are accurate enough for daily summaries | They are classifier outputs with known error |
| Students carried their study phone throughout the day | Days where they did not would understate behaviour |
| Student-days with sufficient data are representative of the term | If missing data is concentrated in stressful weeks, high-stress days are under-represented |

<br>

**Risks**

| Risk | Why it matters | Mitigation |
| --- | --- | --- |
| Sparse stress responses | Students did not answer everyday, meaning fewer labelled rows | Drop or flag students with too few responses |
| Missing sensor data | Phone off, uncharged, or not carried | Exclude days with too little data |
| Ethics and Privacy | Behavioural and mental health data about real people | Frame the findings as patterns, not individual labels |

### **1.7 Initial Project Plan**

**Proposed Technique:** 

Binary classification using a decision tree, compared against a majority class baseline and an alternative model, evaluated with recall and F1 on a student level split.

<br>

**Tools:** 

Python, Jupyter Notebook, Github

<br>

**CRISP-DM Activities**

| Phase | Activities |
| --- | --- |
| *Business Understanding* | Problem, stakeholders, objectives, success criteria, constraints |
| *Data Understanding* | Feasibility check (stress responses per student, sensor coverage), data description, exploration and quality report, data dictionary |
| *Data Preparation* | Recode stress scale, aggregate sensor data to student-days, join with stress labels, handle missing data, preparation log, before/after validation |
| *Modelling* | Student-level train/test split, baseline, decision tree |
| *Evaluation* | Metrics against success criteria, error analysis, limitations, bias and ethics |
| *Deployment & submission* | Recommendations and monitoring, reflection, AI collaboration statement, final paper |


*The plan is to be revised as the project develops, with changes and their reasoning recorded accordingly*

## **Phase 2: Data Understanding**

### **Enviornment Setup**

In [7]:
# data wrangling
import pandas as pd
import numpy as np
from pathlib import Path

# data viz
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


### **2.1 Initial Data Collection Report**

**2.1.1 Source and Permissions**

| Type | Details |
| --- | --- |
| Data Source | Kaggle dataset Studentlife https://www.kaggle.com/datasets/dartweichen/student-life?select=dataset |
| Owner / Publisher | Studentlife Dartmouth College (Wang et al., 2014) – https://studentlife.cs.dartmouth.edu/ |
| Access Date | 29 October 2026 |
| Licence / Permission | No license explicitly stated on the Kaggle dataset page |

<br>

**2.1.2 Collection Process**

| Type | Details |
| --- | --- |
| Collection Process | A StudentLife android app passively collected sensor based data such as activity, sleep, and location, and delivered EMA surveys. Pre and post term surveys were conducted via SurveyMonkey. |

<br>

**2.1.3 Population and Scope**

| Type | Details |
| --- | --- |
| Population | 48 Dartmouth students who completed the study |
| Time Period | Spring term 2013, for a period of 10 weeks  |
| Geography | Dartmouth College, New Hampshire, USA. |
| Scope | **Dataset:** passive sensing inferences (activity, conversation, phone lock/charge, light, location), EMA responses (stress, mood, sleep, social and others, EMA and mood responses), pre/post mental health surveys, and education data (grades, deadlines) – 52.6 GB of sensing data in total (Wang et al., 2014). **This project:** stress EMA (target), phone lock, activity and one sleep measure, with all other data excluded. |

<br>

**2.1.4 Exclusions and Limitations**

| Type | Details |
| --- | --- |
| Known Exclusions | Students who dropped out of the study or the class were removed (60 joined, 48 completed). |
| Aggregation | The sensor data are inferences from phone classifiers, not raw sensor readings. |
| Capping | None identified. |
| Sampling | Voluntary sample from one class, not random. |
| Access Limitations | Kaggle repost of the original |

<br>

**2.1.5 Files Acquired**

To what extent can passively sensed smartphone usage and sleep behaviour distinguish days on which university students report elevated stress from days on which they do not?

| Required from Question | File / folder | Contents | Key fields | How it relates |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

### **2.2 Data Description Report**

### **2.3 Data Exploration Report**

### **2.4 Data Quality Report**